In this notebook, we will test an agent that queries JSON data using natural language by translating the query into JavaScript. For a given question related to the data in a JSON file, the results of the agent’s queries will be compared to those of manually written queries.

In [1]:
%pip install jsonschema
%pip install langchain
%pip install langchain_mistralai
%pip install pandas
%pip install aijsondbpy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


First, we create the agent. For more details, see the sample notebook `single_js.ipynb`.

In [2]:
import aijsondb
from langchain.tools import tool
import json
from langchain_mistralai.chat_models import ChatMistralAI
from langchain.agents import create_agent

aijsondb.init_db("./data/500 KB_V3.json","./data/employeeSchemaDescription_V3.json")
last_answer_from_tool=None
@tool
def query_json_javascript(query: str) -> any:
    """Run a javascript expression on the provided data."""
    global last_answer_from_tool
    try:
        last_answer_from_tool=matches=None
        matches=aijsondb.query_data_javascript(query)
        last_answer_from_tool=matches
        return matches
    except Exception as e:  
        serr=f"Error running JSONPath query: {e}"
        return serr 
    
with open('../data/talktodata/employeeSchemaDescription_V3.json') as f:
    jschema = json.load(f)  

schema = json.dumps(jschema, indent=4)

template =f"""
You are a JavaScript query assistant for a JSON data object. Your task is to generate a JavaScript expression to fetch data from the object based on a natural language question.

### JSON Schema:
The data follows this schema:
{schema}

### Rules:
1. **Data Access**:
   - The JSON object is stored in a variable named `data`.
   - The query result must be assigned to a variable named `result` (e.g., `var result = ...`).

2. **Query Generation**:
   - Only generate a JavaScript expression if the question can be answered using the provided schema.
   - If the question **cannot** be answered (e.g., the data does not exist in the schema or the question is irrelevant), respond with:
     *"This question cannot be answered with the available data."*

3. **Tool Usage**:
   - If the question is valid, use the tool:
     `query_json_javascript` (returns data using the generated JavaScript query).
   - Do **not** call the tool if the question is unanswerable.

"""    

llmm = ChatMistralAI(
    model="mistral-large-latest",
    temperature=0,
    max_retries=3,
)

agent = create_agent(
    model=llmm,
    tools=[query_json_javascript],
    system_prompt=template,
)

The result of the agent invocation is the full agent history. The last message in this history is the final answer provided by the agent.

The message immediately before the final answer is a `ToolMessage`, which contains the data fetched by the query from the JSON file. We use this message to compare it with the results of the corresponding manually written query as a reference.

For this purpose, we create a convenience function called `get_answer`. The return value of this function includes the parameter `"answer"`, which contains the question in natural language. The `"isOk"` property of the return value indicates whether the tool call was successful.

In [3]:
from langchain.messages import ToolMessage 

def checkAgentCall(output):
    if len(output['messages'])>=4 and type(output['messages'][-2]) is ToolMessage:
        aiMessage=output["messages"][-1]
        modelAnswer=""
        if aiMessage != None and hasattr(aiMessage, 'content') and aiMessage.content is not None:
            if type(aiMessage.content) == str:
                modelAnswer=aiMessage.content.strip()
            else:
                modelAnswer=f"Error:{str(aiMessage.content)}"
        else:
            modelAnswer="No data!"
        return{
            "firstTry":len(output['messages'])==4,
            "isOk": type(output['messages'][-2]) is ToolMessage,
            "answer": output['messages'][-2].content,
            "query" : output['messages'][-3].tool_calls[0]['args']['query'],
            "modelAnswer": modelAnswer
        }
    else:
        return {
         "firstTry": True,
         "isOk": False,
         "answer": "",
         "query" : "",
         "modelAnswer": ""
       }
    

def get_answer(question):
    output=agent.invoke(
      {"messages": [{"role": "user", "content": question}]}
    )
    return checkAgentCall(output)

We can now test whether our function works.

In [4]:
get_answer("Please give me the number of employees.")

{'firstTry': True,
 'isOk': True,
 'answer': '201',
 'query': 'var result = data.employees.length;',
 'modelAnswer': 'The number of employees is **201**.'}

The answer from the tool call is stored as a JSON string in the `answer` property.

In the file `data/test_queries_v3.json`, you will find the reference query for this question, which was written manually. Since it is identical, the comparison is straightforward: `answer == answer_ref`.

In other cases, the result is a JSON string containing an object or an array. To verify the query results, we created the function `compareToRef` in the Python module `comparejs`.

If the expected results are grouped by a property (e.g., employees per state), we use the function `compareGroupToRef` in the Python module `comparejs`.

For more details, refer to the `comparejs` module. Here, we simply call the test.

In [5]:
import comparejs
comparejs.testCompareToRef()

True

Our test cases are organized into four files containing JSON data:

- **test_queries_v3.json**: Basic test cases with questions of varying complexity.
- **test_queries_no.json**: None; questions that cannot be answered or yield no result.
- **test_queries_va.json**: Variant; variations of basic questions where the reference result remains the same.
- **test_queries_ex.json**: Extra; additional questions with relatively low complexity.

Each file contains an array of test cases. A test case is structured as follows:

```json
{
    "question": "Please give me the number of employees.",
    "js_query_ref_type": "js",
    "js_query_ref": "var result = data.employees.length;",
    "sql_query_ref": "SELECT COUNT(*) FROM employees",
    "type": "count"
}
```

- **question**: The question about the test data, expressed in natural language.
- **js_query_ref**: The reference JavaScript query to fetch the data.
- **sql_query_ref**: The reference SQL query to fetch the data.
- **type**: The type of query.

For the benchmark, we first use our agent to fetch the data based on the text in `question`. Then, we fetch the data using the JavaScript query `js_query_ref`. Both results are compared using the function `compareToRef`. If the comparison is successful, the test case is marked with `answerOK[i] = 1`; otherwise, it is marked with `answerOK[i] = 0`. The comparison results are finally saved in a pandas DataFrame.







In [6]:
import pandas as pd

questions=[]
answerOK=[]
category=[]
lastAnswerModel=None
lastAnswerRef=None


# Writing text to a file (overwrites if exists)
with open("log_js.txt", "w", encoding="utf-8") as f:
   f.write("Log\n")

def add_log(line):
   with open("log_js.txt", "a", encoding="utf-8") as f:
      f.write(str(line)+"\n")


def check_answer(questions, answerOK, add_log, qa):
    global last_answer_from_tool
    last_answer_from_tool=None
    q=qa["question"]
    qtype=qa["type"]
    jsRefType=qa["js_query_ref_type"]
    isJS=False
    jpath_answer=None
    js_answer=None
    jref=None 
    no_tool_call=False
    if 'js' == jsRefType:
       js_answer=qa["js_query_ref"]
       if len(js_answer)==0:
          no_tool_call=True
       else:
          jref=aijsondb.query_data_javascript(js_answer)
    else:
       raise Exception("jmespath not supported yet")
    add_log(f"Question: {q}")
    answer=get_answer(q)
    add_log(f"Answer: {answer}")
    copo=False
    janswer_global=last_answer_from_tool
    add_log(f"Global Answer: {janswer_global}")
    janswer=last_answer_from_tool
    query=answer['query']
    add_log(query)
    add_log("Mod--------\n")
    add_log(type(janswer))
    add_log(janswer)
    add_log(type(janswer) is list)
    add_log("Ref--------\n")
    add_log(type(jref))
    add_log(jref)
    add_log(type(jref) is list)
    if no_tool_call == False:      
      if qtype == "group":
          copo=comparejs.compareGroupToRef(janswer,jref)
      else:
          copo=comparejs.compareToRef(janswer,jref)
      add_log(str(copo))
    else:
       sanswer=answer['answer']
       if len(sanswer)==0:
          copo=True

    iOKAnswer=0
    if copo:
        iOKAnswer=1
    answerOK.append(iOKAnswer)
    questions.append(q)

We start with the category basic test cases.

In [8]:
import time
with open('data/test_queries_v3.json') as f:
    testqueries = json.load(f) 

test_data=testqueries['data']

for qa in test_data[0:25]:
    for ity in range(3):
        try:
            check_answer(questions, answerOK, add_log, qa)
            category.append('yes')
            time.sleep(5)
            break
        except Exception as e:
            add_log('Error: '+str(e))

None test cases.

In [9]:
import time
qs=questions.copy()
aok=answerOK.copy()
cat=category.copy()

with open('data/test_queries_no.json') as f:
    testqueries = json.load(f) 

test_data=testqueries['data']

for qa in test_data[0:25]:
    for ity in range(3):
        try:
            check_answer(questions, answerOK, add_log, qa)
            category.append('no')
            time.sleep(5)
            break
        except Exception as e:
            add_log('Error: '+str(e))

Variant test cases.

In [ ]:
import time
qs=questions.copy()
aok=answerOK.copy()
cat=category.copy()

with open('data/test_queries_va.json') as f:
    testqueries = json.load(f) 

test_data=testqueries['data']

for qa in test_data[0:25]:
    for ity in range(3):
        try:
            check_answer(questions, answerOK, add_log, qa)
            category.append('va')
            time.sleep(5)
            break
        except Exception as e:
            add_log('Error: '+str(e))

Extra test cases.

In [12]:
qs=questions.copy()
aok=answerOK.copy()
cat=category.copy()

with open('data/test_queries_ex.json') as f:
    testqueries = json.load(f) 

test_data=testqueries['data']

for qa in test_data[0:25]:
    for ity in range(3):
        try:
           check_answer(questions, answerOK, add_log, qa)
           category.append('ex')
           time.sleep(5)
           break
        except Exception as e:
            add_log('Error: '+str(e))

Now that we have the results in the `questions`, `answerOK`, and `category` lists, we can create a DataFrame and save it for future reference.

In [13]:
df = pd.DataFrame({
  "questions":questions,
  "okAnswer": answerOK,
  "category": category
})
df.to_pickle("data/result_js.pkl")


For a comparison with the corresponding SQL queries, see the notebook `comparison.ipynb`. Here, we will only provide the percentage of correct answers.

In [14]:
df['okAnswer'].sum()/len(df['okAnswer'])*100

np.float64(90.0)